In [ ]:
#!git clone https://github.com/modelscope/DiffSynth-Studio.git
#%cd DiffSynth-Studio

#!pip install -q -e ".[quant]"
#!pip install -q -U bitsandbytes accelerate pillow pandas

Cloning into 'DiffSynth-Studio'...
remote: Enumerating objects: 16339, done.
remote: Counting objects: 100% (6658/6658), done.jects:  83% (5527/6658)
remote: Compressing objects: 100% (1634/1634), done.
remote: Total 16339 (delta 5375), reused 5024 (delta 5024), pack-reused 9681 (from 2)
Receiving objects: 100% (16339/16339), 19.61 MiB | 8.81 MiB/s, done.
Resolving deltas: 100% (11303/11303), done.
/Users/fernandodavilalbcfilho/Downloads/aquario/notebooks/DiffSynth-Studio/DiffSynth-Studio


/Users/fernandodavilalbcfilho/Library/Python/3.9/lib/python/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


ERROR: File "setup.py" or "setup.cfg" not found. Directory cannot be installed in editable mode: /Users/fernandodavilalbcfilho/Downloads/aquario/notebooks/DiffSynth-Studio/DiffSynth-Studio
(A "pyproject.toml" file was found, but editable mode currently requires a setuptools-based build.)
You should consider upgrading via the '/Users/fernandodavilalbcfilho/Downloads/aquario/.venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/fernandodavilalbcfilho/Downloads/aquario/.venv/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
MPS_AVAILABLE = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

print("PyTorch:", torch.__version__)
print("CUDA disponível:", CUDA_AVAILABLE)
print("MPS disponível:", MPS_AVAILABLE)

if CUDA_AVAILABLE:
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
elif MPS_AVAILABLE:
    DEVICE = "mps"
    DTYPE = torch.float16
    print("Dispositivo selecionado: Apple Metal (MPS)")
    print("Aviso: este notebook usa DiffSynth-Studio, cujo carregador Qwen suporta oficialmente CUDA/CPU, não MPS.")
    print("Além disso, Qwen-Image tem cerca de 20B parâmetros e este Mac possui 24 GB de memória unificada.")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("Nenhum acelerador disponível; dispositivo selecionado: CPU")

if DEVICE != "cuda":
    raise RuntimeError(
        "Este notebook DiffSynth precisa de uma GPU NVIDIA/CUDA para carregar o Qwen-Image. "
        "No Mac, use o dry-run ou execute em Colab/Linux com CUDA; MPS não é suportado por este notebook e 24 GB não são suficientes para este modelo."
    )


PyTorch: 2.8.0
CUDA disponível: False
MPS disponível: True
Dispositivo selecionado: Apple Metal (MPS)
Aviso: este notebook usa DiffSynth-Studio, cujo carregador Qwen suporta oficialmente CUDA/CPU, não MPS.
Além disso, Qwen-Image tem cerca de 20B parâmetros e este Mac possui 24 GB de memória unificada.


RuntimeError: Este notebook DiffSynth precisa de uma GPU NVIDIA/CUDA para carregar o Qwen-Image. No Mac, use o dry-run ou execute em Colab/Linux com CUDA; MPS não é suportado por este notebook e 24 GB não são suficientes para este modelo.

In [ ]:
PROJECT_ROOT = "/caminho/para/pixel_project"

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/fernandodavilalbcfilho/Downloads/aquario")
CSV_PATH = PROJECT_ROOT / "kauar_peixes.csv"

ORIGINALS = PROJECT_ROOT / "data" / "sources"
CANDIDATES = PROJECT_ROOT / "output" / "candidates"
TARGETS = PROJECT_ROOT / "targets"
DATASET = PROJECT_ROOT / "dataset"
INPUT_FOLDER = ORIGINALS
OUTPUT_FOLDER = PROJECT_ROOT / "output" / "testee"

for folder in [ORIGINALS, CANDIDATES, TARGETS, DATASET, OUTPUT_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"CSV: {CSV_PATH}")
print(f"Fotos: {ORIGINALS}")
print(f"Saídas: {OUTPUT_FOLDER}")


In [ ]:
import os
import torch

from diffsynth.pipelines.qwen_image import (
    QwenImagePipeline,
    ModelConfig
)

os.environ["DIFFSYNTH_DOWNLOAD_SOURCE"] = "huggingface"

pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=DTYPE,
    device=DEVICE,
    model_configs=[
        ModelConfig(
            model_id="Qwen/Qwen-Image-Edit-2511",
            origin_file_pattern="transformer/diffusion_pytorch_model*.safetensors",
        ),
        ModelConfig(
            model_id="Qwen/Qwen-Image",
            origin_file_pattern="text_encoder/model*.safetensors",
        ),
        ModelConfig(
            model_id="Qwen/Qwen-Image",
            origin_file_pattern="vae/diffusion_pytorch_model.safetensors",
        ),
    ],
    processor_config=ModelConfig(
        model_id="Qwen/Qwen-Image-Edit",
        origin_file_pattern="processor/",
    ),
)

print("Modelo carregado!")

In [ ]:
PIXEL_PROMPT = """
Transform Figure 1 into clean handcrafted pixel art.

Preserve:
- the original composition
- identity of the subjects
- pose
- proportions
- clothing
- important objects
- camera angle
- relative position of every element

Visual style:
- authentic 16-bit pixel art
- clearly visible square pixels
- hard pixel edges
- no anti-aliasing
- limited color palette
- deliberate pixel clusters
- simplified shapes
- crisp silhouettes
- retro videogame sprite aesthetic
- consistent pixel size throughout the image
- no painterly texture
- no smooth digital brush strokes
- no photorealism
- no blur

Do not add new objects.
Do not change the composition.
"""

In [ ]:
from PIL import Image
from pathlib import Path

image_path = next(ORIGINALS.glob("*"))

source = Image.open(image_path).convert("RGB")

result = pipe(
    prompt=PIXEL_PROMPT,
    edit_image=[source],
    seed=42,
    num_inference_steps=40,
    height=1024,
    width=1024,
    edit_image_auto_resize=True,
    zero_cond_t=True,
)

result.save(PROJECT_ROOT / "teste_pixel.png")

display(source)
display(result)

In [ ]:
from PIL import Image

def enforce_pixel_grid(
    image,
    logical_resolution=128,
    output_resolution=1024
):
    image = image.convert("RGB")

    small = image.resize(
        (logical_resolution, logical_resolution),
        Image.Resampling.LANCZOS
    )

    pixel = small.resize(
        (output_resolution, output_resolution),
        Image.Resampling.NEAREST
    )

    return pixel

In [ ]:
pixel_final = enforce_pixel_grid(
    result,
    logical_resolution=128,
    output_resolution=1024,
)

display(pixel_final)

pixel_final.save(
    PROJECT_ROOT / "teste_pixel_grid.png"
)

In [ ]:
def limit_palette(image, colors=64):
    return image.quantize(
        colors=colors,
        method=Image.Quantize.MEDIANCUT
    ).convert("RGB")

In [ ]:
pixel_final = enforce_pixel_grid(result)

pixel_final = limit_palette(
    pixel_final,
    colors=64
)

display(pixel_final)

In [ ]:
#colors=16
#colors=32
#colors=64
#colors=128

In [ ]:
from pathlib import Path
from PIL import Image
import torch

extensions = {
    ".jpg", ".jpeg", ".png", ".webp"
}

files = [
    f for f in ORIGINALS.iterdir()
    if f.suffix.lower() in extensions
]

print(f"{len(files)} imagens encontradas.")

In [ ]:
SEEDS = [11, 42, 73]

for index, path in enumerate(files):

    source = Image.open(path).convert("RGB")

    print(
        f"[{index+1}/{len(files)}]",
        path.name
    )

    for seed in SEEDS:

        output = pipe(
            prompt=PIXEL_PROMPT,
            edit_image=[source],
            seed=seed,
            num_inference_steps=40,
            height=1024,
            width=1024,
            edit_image_auto_resize=True,
            zero_cond_t=True,
        )

        output = enforce_pixel_grid(
            output,
            logical_resolution=128,
            output_resolution=1024,
        )

        output = limit_palette(
            output,
            colors=64,
        )

        output_path = (
            CANDIDATES /
            f"{path.stem}_seed_{seed}.png"
        )

        output.save(output_path)

print("Finalizado.")

In [ ]:
import shutil
import json

dataset_original = DATASET / "original"
dataset_pixel = DATASET / "pixel"

dataset_original.mkdir(exist_ok=True)
dataset_pixel.mkdir(exist_ok=True)

records = []

targets = list(TARGETS.glob("*.png"))

for target in targets:

    source_candidates = list(
        ORIGINALS.glob(target.stem + ".*")
    )

    if not source_candidates:
        print("Sem original:", target.name)
        continue

    source = source_candidates[0]

    source_name = target.stem + ".png"
    target_name = target.stem + ".png"

    # normaliza original para PNG
    source_image = Image.open(source).convert("RGB")
    source_image.save(
        dataset_original / source_name
    )

    target_image = Image.open(target).convert("RGB")
    target_image.save(
        dataset_pixel / target_name
    )

    records.append({
        "image": f"pixel/{target_name}",
        "edit_image": f"original/{source_name}",
        "prompt": (
            "Convert Figure 1 into the PIXELFERN style, "
            "preserving composition, identity, pose and objects."
        )
    })

metadata_path = DATASET / "metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    f"{len(records)} pares criados."
)

print(metadata_path)

In [ ]:
import json

with open(DATASET / "metadata.json") as f:
    metadata = json.load(f)

metadata[:3]

In [ ]:
DATASET_PATH = str(DATASET)
OUTPUT_PATH = str(PROJECT_ROOT / "pixel_lora")

In [ ]:
!accelerate launch examples/qwen_image/model_training/train.py \
    --dataset_base_path "$DATASET_PATH" \
    --dataset_metadata_path "$DATASET_PATH/metadata.json" \
    --data_file_keys "image,edit_image" \
    --extra_inputs "edit_image" \
    --max_pixels 262144 \
    --dataset_repeat 20 \
    --model_id_with_origin_paths "Qwen/Qwen-Image-Edit-2511:transformer/diffusion_pytorch_model*.safetensors,Qwen/Qwen-Image:text_encoder/model*.safetensors,Qwen/Qwen-Image:vae/diffusion_pytorch_model.safetensors" \
    --learning_rate 1e-4 \
    --num_epochs 5 \
    --remove_prefix_in_ckpt "pipe.dit." \
    --output_path "$OUTPUT_PATH" \
    --lora_base_model "dit" \
    --lora_target_modules "to_q,to_k,to_v,add_q_proj,add_k_proj,add_v_proj,to_out.0,to_add_out,img_mlp.net.2,img_mod.1,txt_mlp.net.2,txt_mod.1" \
    --lora_rank 32 \
    --use_gradient_checkpointing \
    --gradient_accumulation_steps 4 \
    --dataset_num_workers 2 \
    --find_unused_parameters \
    --zero_cond_t

In [ ]:
!accelerate launch examples/qwen_image/model_training/train.py \
    --dataset_base_path "$DATASET_PATH" \
    --dataset_metadata_path "$DATASET_PATH/metadata.json" \
    --data_file_keys "image,edit_image" \
    --extra_inputs "edit_image" \
    --max_pixels 262144 \
    --dataset_repeat 20 \
    --model_id_with_origin_paths "Qwen/Qwen-Image-Edit-2511:transformer/diffusion_pytorch_model*.safetensors,Qwen/Qwen-Image:text_encoder/model*.safetensors,Qwen/Qwen-Image:vae/diffusion_pytorch_model.safetensors" \
    --quant_options "Qwen/Qwen-Image-Edit-2511:transformer/diffusion_pytorch_model*.safetensors:bitsandbytes_nf4;Qwen/Qwen-Image:text_encoder/model*.safetensors:bitsandbytes_nf4" \
    --learning_rate 1e-4 \
    --num_epochs 5 \
    --remove_prefix_in_ckpt "pipe.dit." \
    --output_path "$OUTPUT_PATH" \
    --lora_base_model "dit" \
    --lora_target_modules "to_q,to_k,to_v,add_q_proj,add_k_proj,add_v_proj,to_out.0,to_add_out,img_mlp.net.2,img_mod.1,txt_mlp.net.2,txt_mod.1" \
    --lora_rank 32 \
    --use_gradient_checkpointing \
    --gradient_accumulation_steps 4 \
    --dataset_num_workers 2 \
    --find_unused_parameters \
    --zero_cond_t

In [ ]:
import torch
from PIL import Image

from diffsynth.pipelines.qwen_image import (
    QwenImagePipeline,
    ModelConfig,
)

pixel_pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device="cuda",

    model_configs=[
        ModelConfig(
            model_id="Qwen/Qwen-Image-Edit-2511",
            origin_file_pattern="transformer/diffusion_pytorch_model*.safetensors",
        ),

        ModelConfig(
            model_id="Qwen/Qwen-Image",
            origin_file_pattern="text_encoder/model*.safetensors",
        ),

        ModelConfig(
            model_id="Qwen/Qwen-Image",
            origin_file_pattern="vae/diffusion_pytorch_model.safetensors",
        ),
    ],

    processor_config=ModelConfig(
        model_id="Qwen/Qwen-Image-Edit",
        origin_file_pattern="processor/",
    ),
)

In [ ]:
LORA_PATH = (
    PROJECT_ROOT /
    "pixel_lora" /
    "epoch-4.safetensors"
)

pixel_pipe.load_lora(
    pixel_pipe.dit,
    str(LORA_PATH),
)

print("Pixel LoRA carregado!")

In [ ]:
new_image = Image.open(
    PROJECT_ROOT / "nova_imagem.jpg"
).convert("RGB")

prompt = """
Convert Figure 1 into PIXELFERN style.

Preserve exactly:
identity,
pose,
composition,
clothing,
objects,
camera angle.

Authentic pixel art.
Hard pixel edges.
Limited palette.
No antialiasing.
No blur.
"""

output = pixel_pipe(
    prompt=prompt,
    edit_image=[new_image],
    seed=42,
    num_inference_steps=40,
    height=1024,
    width=1024,
    edit_image_auto_resize=True,
    zero_cond_t=True,
)

display(output)

In [ ]:
output = enforce_pixel_grid(
    output,
    logical_resolution=128,
    output_resolution=1024,
)

output = limit_palette(
    output,
    colors=64,
)

display(output)

output.save(
    PROJECT_ROOT / "nova_imagem_pixel.png"
)

In [ ]:
import csv
import hashlib
from urllib.request import Request, urlopen
from PIL import Image
from io import BytesIO

LIMIT = 1  # Use None para processar todas as linhas elegíveis.
START = 0


def download_csv_image(row):
    image_url = (row.get("imagem_principal_url") or "").strip()
    if not image_url:
        raise ValueError("A linha não possui imagem_principal_url")

    item_id = hashlib.sha256(
        (row.get("url", image_url)).encode("utf-8")
    ).hexdigest()[:16]
    image_path = ORIGINALS / f"{item_id}.png"

    if not image_path.exists():
        request = Request(image_url, headers={"User-Agent": "aquario-pixelart/1.0"})
        with urlopen(request, timeout=60) as response:
            image = Image.open(BytesIO(response.read())).convert("RGB")
        image.save(image_path)

    return image_path, item_id


with CSV_PATH.open(newline="", encoding="utf-8") as csv_file:
    rows = list(csv.DictReader(csv_file))

rows = rows[START:]
if LIMIT is not None:
    rows = rows[:LIMIT]

print(f"{len(rows)} linha(s) selecionada(s) de {CSV_PATH.name}.")

for index, row in enumerate(rows, START + 1):
    try:
        image_path, item_id = download_csv_image(row)
        species = row.get("nome_cientifico") or row.get("nome_popular") or "aquarium organism"
        common_name = row.get("nome_popular") or species
        prompt = f"""Create a clean 16-bit pixel art aquarium sprite of {common_name} ({species}), preserving the real animal or plant silhouette, main colors, distinctive markings, fins, antennae, leaves or roots. Single isolated subject, orthographic side view for fish and shrimp, upright natural pose for aquarium plants, white studio background, crisp pixel clusters, limited palette, dark navy 1 pixel outline, no scenery, no substrate, no bubbles, no text, centered with comfortable margins."""

        print(f"[{index}] {common_name}: {image_path.name}")
        source = Image.open(image_path).convert("RGB")

        output = pixel_pipe(
            prompt=prompt,
            edit_image=[source],
            seed=42,
            num_inference_steps=40,
            height=1024,
            width=1024,
            edit_image_auto_resize=True,
            zero_cond_t=True,
        )

        output = enforce_pixel_grid(
            output,
            logical_resolution=128,
            output_resolution=1024,
        )
        output = limit_palette(output, colors=64)
        output.save(OUTPUT_FOLDER / f"{item_id}.png")
        print(f"Salvo em: {OUTPUT_FOLDER / f'{item_id}.png'}")
    except Exception as exc:
        print(f"Falha na linha {index}: {exc}")

print("Finalizado.")
